# Titanic: train ve test ile tahmin

Bu notebook, ekran görüntüsündeki Kaggle dosyalarını okur.

- `train.csv`: 891 yolcu, `Survived` etiketi var
- `test.csv`: 418 yolcu, etiket yok

Kadın yaşar / erkek ölür kuralı train üzerinde yaklaşık **%78.7** doğruluk verir. Aşağıdaki özellikler ve model karşılaştırması, 5 katmanlı cross-validation ile bunun üstüne çıkmayı hedefler. Beklenen bant yaklaşık **%80–83**. Seçilen model tüm train ile yeniden eğitilir ve `output/submission.csv` yazılır.

Kaggle’a bu dosyayı yükle. Public skor, cross-validation sonucunun bir iki puan altında kalabilir.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

for candidate in [Path.cwd(), Path.cwd() / "titanic"]:
    if (candidate / "pipeline.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

import pipeline

train, test = pipeline.load_data()
print(f"train {train.shape}  test {test.shape}")
train.head()

In [ ]:
test.head()

## Özellikler

Model ham isim ve kabin metnini görmez. Satırdan şu alanlar üretilir:

- **Title** — `Master` erkek çocukları `Mr` grubundan ayırır
- **FamilySize / IsAlone** — 2–4 kişilik aileler daha sık hayatta kalır
- **TicketGroup / FarePerPerson** — aynı bileti paylaşanlar ve bilet ücretinin kişi başı hali
- **Age, AgeBand, IsChild** — eksik yaş, aynı ünvan ve sınıfın medyanı ile dolar
- **Deck / HasCabin** — kabin harfi; boş kabin ayrı kategori olarak kalır
- **Sex, Pclass, Embarked**

Yaş, ücret ve liman istatistikleri her cross-validation katında yalnızca o katmanın train parçasına fit edilir. Bilet sayısı etiketsizdir; train ve test birlikte sayılır.

In [ ]:
pipeline.preview_features(train, test)

## Model seçimi

Adaylar: lojistik regresyon, random forest, gradient boosting, histogram gradient boosting ve bunların soft vote birleşimi. Metrik accuracy. En yüksek ortalama doğruluğu veren model gönderiyi üretir.

In [ ]:
report = pipeline.run()
report.scores

In [ ]:
report.oof_by_group

## Gönderi

`Survived` burada tahmindir. Test etiketleri dosyada yoktur. Kaggle’a `titanic/output/submission.csv` yüklenir.

In [ ]:
report.submission.head(10)